# Getting started with PASSAGE

This notebook is the shortest path from the dataset to a training example. It loads one released
scene, decodes its input tensor, draws a reference route and turns that route into a segmentation
target. Every step uses only the public `passage` package and the data.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads one row and runs in a few seconds.

Without the code repository, the same row is one `datasets` call away:

```python
from datasets import load_dataset
rows = load_dataset("thalesgroup/passage", "256x256_samples", split="validation",
                    revision="2259dd94e2de28ab4a56c270247a7e347ec8f133")
```

In [ ]:
resolution = 256
split = "validation"
sample_id = None  # None takes the first scene whose routes span a quarter of the grid side
output_dir = "notebook-artifacts/notebooks/demo"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. Load one scene

A row holds the compressed tensor, the six reference routes and a metadata record. The metadata
describes where the crop comes from, how the tensor was normalised and what each route costs.

In [ ]:
row = source.get_row(resolution, split, sample_id=sample_id or first_scene(resolution, split), samples=False)
meta = row["metadata"]
overview = pd.DataFrame([
    {"Field": "Scene ID", "Value": row["id"]},
    {"Field": "Configuration", "Value": f"{resolution}×{resolution}, {split} split"},
    {"Field": "Crop centre", "Value": "{:.3f}° {}, {:.3f}° {}".format(
        abs(meta["crop"]["center"]["lat"]), "N" if meta["crop"]["center"]["lat"] >= 0 else "S",
        abs(meta["crop"]["center"]["lon"]), "E" if meta["crop"]["center"]["lon"] >= 0 else "W")},
    {"Field": "Calibration", "Value": f"{meta['calibration']['min']:,.0f} m to {meta['calibration']['max']:,.0f} m"},
    {"Field": "Obstacle coverage", "Value": f"{100 * meta['obstacles']['ratio_actual']:.1f} % of cells"},
    {"Field": "Start (row, column)", "Value": f"({meta['markers']['start_point']['i']}, {meta['markers']['start_point']['j']})"},
    {"Field": "Goal (row, column)", "Value": f"({meta['markers']['end_point']['i']}, {meta['markers']['end_point']['j']})"},
    {"Field": "Stored routes", "Value": ", ".join(k for k in row if k.startswith("path_"))},
])
show(overview, "Table 1. The main metadata fields of the scene.")

## 2. Decode the tensor

`decode_scene` decompresses the tensor, multiplies the elevation channel back to metres with the
recorded calibration, and returns the obstacle mask and the two endpoints as (row, column) cells.

In [ ]:
elevation, obstacles, start, goal = decode_scene(row)
arrays = pd.DataFrame([
    {"Array": "elevation", "Shape": str(elevation.shape), "Type": str(elevation.dtype),
     "Minimum": float(elevation.min()), "Maximum": float(elevation.max())},
    {"Array": "obstacles", "Shape": str(obstacles.shape), "Type": str(obstacles.dtype),
     "Minimum": float(obstacles.min()), "Maximum": float(obstacles.max())},
])
show(arrays, "Table 2. Decoded arrays.", "{:,.1f}")

## 3. Draw a reference route

Each route is an ordered list of (row, column) cells from start to goal. Figure 1 draws the
energy-optimal route with the obstacles blocked, in the colour the paper uses for the energy objective.

In [ ]:
from matplotlib.lines import Line2D

route = np.asarray(row["path_astar_energy_obstacles"])
outcome = meta["paths"]["astar_energy"]["obstacles"]
fig = render_scene(elevation, obstacles, start, goal, {"energy": route}, legend=False,
                   title=f"Scene {row['id']} · {resolution}×{resolution}")
fig.axes[0].lines[0].set_color(OBJECTIVE_COLORS["energy"])
fig.legend(handles=[Line2D([], [], color=OBJECTIVE_COLORS["energy"], lw=1.8, label="Energy route, obstacles blocked")]
           + comparison_handles()[2:], loc="lower center", ncol=4, fontsize=8, frameon=False)
figures = save_figure(fig, output_path, f"demo-{resolution}-{row['id']}")
plt.show()
display(Markdown(f"Status **{outcome['status']}** · {len(route)} cells · stored cost {outcome['cost']:,.1f}."))

*Figure 1. One scene with its energy-optimal route. Hatched cells are obstacles; the white circle is
the start and the white star the goal.*

## 4. Turn the route into a training target

A segmentation model learns a mask with 1 on the route cells. Every released route joins the start to
the goal, so every variant gives a target.

In [ ]:
from matplotlib.colors import ListedColormap

path_mask = np.zeros((resolution, resolution), dtype=np.uint8)
path_mask[route[:, 0], route[:, 1]] = 1
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.0))
fig.subplots_adjust(top=0.8, wspace=0.1)
axes[0].imshow(path_mask, cmap=ListedColormap(["#f4f3ee", "#a32b8e"]), interpolation="nearest")
axes[0].set_title("Path mask (target)")
layers = np.where(path_mask == 1, 2, np.where(obstacles, 1, 0))
axes[1].imshow(layers, cmap=ListedColormap(["#f4f3ee", OBSTACLE_COLOR, "#a32b8e"]), interpolation="nearest",
               vmin=0, vmax=2)
axes[1].set_title("Mask and obstacles")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
figure_title(fig, "A segmentation target from one route",
             f"{int(path_mask.sum())} route cells out of {resolution * resolution:,} ({100 * path_mask.mean():.2f} %)")
figures += save_figure(fig, output_path, f"demo-{resolution}-{row['id']}-target")
plt.show()

*Figure 2. Left: the path mask built from the route. Right: the same mask over the obstacles. Route
cells are a tiny share of the grid, so segmentation losses usually need class weighting.*

## Summary

In [ ]:
display(Markdown("\n".join([
    f"- Scene {row['id']} decodes to a {resolution}×{resolution} elevation grid spanning "
    f"{elevation.max() - elevation.min():,.0f} m, with {100 * obstacles.mean():.1f} % of cells blocked.",
    f"- Its energy route with obstacles has {len(route)} cells; as a mask it covers "
    f"{100 * path_mask.mean():.2f} % of the grid.",
    "- The other notebooks build on these steps: `costmodel` compares the objectives, `obstacles` "
    "studies the obstacle masks, and `solve` re-solves the labels.",
])))
report = {"resolution": resolution, "split": split, "sample_id": row["id"], "route_cells": int(len(route)),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"demo-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*